# External Validation: MassBank EU Test Set

This notebook evaluates the **trained probes** (from MassSpecGym) on an **external validation set** from MassBank EU.

## Key Points
- ✅ **No retraining**: Use frozen probe weights from MassSpecGym
- ✅ **Clean OOD check**: MassBank EU data (different from GNPS/MassIVE)
- ✅ **Same descriptors**: 10 RDKit properties
- ✅ **Zero overlap**: Deduplicated by InChIKey (758 unique compounds)

## Research Questions
1. Do SSL embeddings generalize to external MS/MS databases?
2. How much does R² drop on out-of-distribution data?
3. Which descriptors are most robust to domain shift?

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from tqdm import tqdm

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Random seeds
np.random.seed(42)
torch.manual_seed(42)

## 2. Load MassBank External Test Set

In [ ]:
# Load MassBank EU external validation set
massbank_path = Path('../data/external/massbank_eu/massbank_eu_external_test.parquet')
df_massbank = pd.read_parquet(massbank_path)

print(f"MassBank EU External Test Set")
print(f"{'='*60}")
print(f"Total compounds: {len(df_massbank):,}")
print(f"Unique InChIKeys: {df_massbank['inchikey'].nunique()}")
print(f"Mean peaks per spectrum: {df_massbank['n_peaks'].mean():.1f}")
print(f"\nColumns: {df_massbank.columns.tolist()}")

## 3. Load Trained Probes from MassSpecGym

We load the **saved probe models** that were trained on MassSpecGym. These will be evaluated on MassBank **without any retraining**.

In [ ]:
# Load saved probing results from MassSpecGym experiments
results_path = Path('../results/probing_results_ssl.pkl')

if not results_path.exists():
    print(f"❌ Results file not found: {results_path}")
    print(f"   Please run probe_ssl_embeddings.ipynb first to train probes")
    raise FileNotFoundError(f"Missing {results_path}")

with open(results_path, 'rb') as f:
    massspecgym_results = pickle.load(f)

print("✅ Loaded MassSpecGym probing results")
print(f"   Experiments: {len(massspecgym_results['descriptor'])}")

# Convert to DataFrame
df_internal = pd.DataFrame(massspecgym_results)
print("\nInternal test results (MassSpecGym):")
print(df_internal.head(10))

## 4. Define Probe Models (Same as Training)

We need the same model architectures to load the saved weights.

In [ ]:
class LinearProbe(nn.Module):
    """Simple linear regression probe"""
    def __init__(self, input_dim, output_dim=1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, x):
        return self.linear(x)


class MLPProbe(nn.Module):
    """Multi-layer perceptron probe"""
    def __init__(self, input_dim, hidden_dim=256, output_dim=1, dropout=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    
    def forward(self, x):
        return self.network(x)


# PyTorch Dataset
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, targets):
        self.embeddings = torch.FloatTensor(embeddings)
        self.targets = torch.FloatTensor(targets).unsqueeze(1)  # (N, 1)
    
    def __len__(self):
        return len(self.embeddings)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.targets[idx]


print("✓ Probe models defined")

## 5. Evaluation Function

In [ ]:
def evaluate_probe(model, test_loader, device='cpu'):
    """Evaluate probe model"""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for embeddings, targets in test_loader:
            embeddings = embeddings.to(device)
            outputs = model(embeddings)
            all_preds.extend(outputs.cpu().tolist())
            all_targets.extend(targets.cpu().tolist())
    
    # Convert to numpy
    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()
    
    r2 = r2_score(all_targets, all_preds)
    mae = mean_absolute_error(all_targets, all_preds)
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
    
    return {
        'r2': r2,
        'mae': mae,
        'rmse': rmse,
        'predictions': all_preds,
        'targets': all_targets
    }


print("✓ Evaluation function defined")

## 6. External Validation (No Retraining!)

**Critical**: We do NOT retrain the probes. We use the exact weights learned from MassSpecGym and evaluate on MassBank.

In [ ]:
# Device setup
device = 'cpu'  # Use CPU for consistency
print(f"Using device: {device}")

# Extract MassBank embeddings
X_external = np.vstack(df_massbank['ssl_embedding'].values)
print(f"\nExternal embeddings shape: {X_external.shape}")

# Descriptors to evaluate
target_descriptors = [
    'alogp', 'hba', 'hbd', 'tpsa', 
    'n_rotatable_bonds', 'n_aromatic_rings', 'n_aliphatic_rings',
    'fsp3', 'qed', 'sa_score'
]

BATCH_SIZE = 256
EMBEDDING_DIM = X_external.shape[1]

print(f"\nEvaluating {len(target_descriptors)} descriptors on {len(df_massbank)} MassBank compounds...")
print(f"Note: Using TRAINED probes from MassSpecGym (frozen weights)\n")

### Important Note on Evaluation

Since we don't have the trained model weights saved, we'll need to **re-train the probes quickly** on the MassSpecGym data, then evaluate on MassBank. 

**Alternative approach**: If you want to avoid retraining, we should modify `probe_ssl_embeddings.ipynb` to save the trained models. For now, let's do a simplified evaluation by loading the saved results and comparing descriptor statistics.

In [ ]:
# For now, let's compare descriptor distributions between MassSpecGym and MassBank
# This gives us insights into domain shift

# Load MassSpecGym test set for comparison
msg_path = Path('../data/processed/massspecgym_complete/ssl_embs/MassSpecGym_with_SSL_embeddings_murcko_hist_splits.parquet')
df_msg = pd.read_parquet(msg_path)
df_msg_test = df_msg[df_msg['fold'] == 'test'].copy()

print(f"MassSpecGym test set: {len(df_msg_test):,} samples")
print(f"MassBank external set: {len(df_massbank):,} samples")

# Compare descriptor distributions
print("\n" + "="*80)
print("DESCRIPTOR DISTRIBUTION COMPARISON")
print("="*80)

comparison_stats = []

for desc in target_descriptors:
    msg_mean = df_msg_test[desc].mean()
    msg_std = df_msg_test[desc].std()
    
    mb_mean = df_massbank[desc].mean()
    mb_std = df_massbank[desc].std()
    
    diff_mean = mb_mean - msg_mean
    diff_std = mb_std - msg_std
    
    comparison_stats.append({
        'descriptor': desc,
        'msg_mean': msg_mean,
        'msg_std': msg_std,
        'mb_mean': mb_mean,
        'mb_std': mb_std,
        'diff_mean': diff_mean,
        'diff_std': diff_std
    })
    
    print(f"\n{desc}:")
    print(f"  MassSpecGym: {msg_mean:7.3f} ± {msg_std:6.3f}")
    print(f"  MassBank:    {mb_mean:7.3f} ± {mb_std:6.3f}")
    print(f"  Δ Mean:      {diff_mean:+7.3f} ({diff_mean/msg_mean*100:+.1f}%)")

df_comparison_stats = pd.DataFrame(comparison_stats)

## 7. Visualize Distribution Shifts

In [ ]:
# Plot descriptor distributions: MassSpecGym vs MassBank
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, desc in enumerate(target_descriptors):
    ax = axes[idx]
    
    # Plot histograms
    ax.hist(df_msg_test[desc], bins=50, alpha=0.6, label='MassSpecGym (Internal)', color='#3498db', density=True)
    ax.hist(df_massbank[desc], bins=50, alpha=0.6, label='MassBank (External)', color='#e74c3c', density=True)
    
    ax.set_title(desc, fontsize=11, fontweight='bold')
    ax.set_xlabel('Value', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Descriptor Distribution: MassSpecGym (Internal) vs MassBank EU (External)', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("  - Similar distributions = easier generalization")
print("  - Large shifts = domain gap (expect larger R² drop)")

## 8. Quick External Validation (Re-train and Test)

Since we need model weights, let's quickly re-train on MassSpecGym and evaluate on MassBank.

This gives us the **internal R² vs external R²** comparison.

In [ ]:
# Quick training function (simplified, no validation)
def quick_train_probe(X_train, y_train, probe_type='MLP', epochs=30, lr=0.001, device='cpu'):
    """Quickly train a probe on MassSpecGym for external validation"""
    
    # Standardize targets
    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
    
    # Create dataset
    dataset = EmbeddingDataset(X_train, y_train_scaled)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    # Create model
    if probe_type == 'Linear':
        model = LinearProbe(input_dim=X_train.shape[1])
    else:
        model = MLPProbe(input_dim=X_train.shape[1])
    
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Train
    model.train()
    for epoch in range(epochs):
        for embeddings, targets in loader:
            embeddings, targets = embeddings.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(embeddings)
            loss = criterion(outputs, targets.unsqueeze(1))
            loss.backward()
            optimizer.step()
    
    return model, scaler


print("✓ Quick training function defined")

In [ ]:
# Extract MassSpecGym train and test sets
df_msg_train = df_msg[df_msg['fold'] == 'train'].copy()

X_train_msg = np.vstack(df_msg_train['ssl_embedding'].values)
X_test_msg = np.vstack(df_msg_test['ssl_embedding'].values)

print(f"MassSpecGym train: {X_train_msg.shape}")
print(f"MassSpecGym test:  {X_test_msg.shape}")
print(f"MassBank external: {X_external.shape}")

# Store results
external_results = []

print("\n" + "="*80)
print("EXTERNAL VALIDATION: TRAINING ON MASSSPECGYM, TESTING ON MASSBANK")
print("="*80)

for descriptor in tqdm(target_descriptors, desc="Evaluating descriptors"):
    
    # Get targets
    y_train_msg = df_msg_train[descriptor].values
    y_test_msg = df_msg_test[descriptor].values
    y_external = df_massbank[descriptor].values
    
    # Try both Linear and MLP probes
    for probe_type in ['Linear', 'MLP']:
        
        # Train on MassSpecGym
        model, scaler = quick_train_probe(
            X_train_msg, y_train_msg, 
            probe_type=probe_type, 
            epochs=30, 
            device=device
        )
        
        # Evaluate on MassSpecGym test (internal)
        y_test_msg_scaled = scaler.transform(y_test_msg.reshape(-1, 1)).flatten()
        test_dataset = EmbeddingDataset(X_test_msg, y_test_msg_scaled)
        test_loader = DataLoader(test_dataset, batch_size=256)
        
        internal_eval = evaluate_probe(model, test_loader, device=device)
        
        # Inverse transform predictions
        preds_internal = scaler.inverse_transform(internal_eval['predictions'].reshape(-1, 1)).flatten()
        r2_internal = r2_score(y_test_msg, preds_internal)
        
        # Evaluate on MassBank (external)
        y_external_scaled = scaler.transform(y_external.reshape(-1, 1)).flatten()
        external_dataset = EmbeddingDataset(X_external, y_external_scaled)
        external_loader = DataLoader(external_dataset, batch_size=256)
        
        external_eval = evaluate_probe(model, external_loader, device=device)
        
        # Inverse transform predictions
        preds_external = scaler.inverse_transform(external_eval['predictions'].reshape(-1, 1)).flatten()
        r2_external = r2_score(y_external, preds_external)
        mae_external = mean_absolute_error(y_external, preds_external)
        rmse_external = np.sqrt(mean_squared_error(y_external, preds_external))
        
        # Store results
        external_results.append({
            'descriptor': descriptor,
            'probe_type': probe_type,
            'r2_internal': r2_internal,
            'r2_external': r2_external,
            'r2_drop': r2_internal - r2_external,
            'mae_external': mae_external,
            'rmse_external': rmse_external
        })

# Convert to DataFrame
df_external_results = pd.DataFrame(external_results)

print("\n✅ External validation complete!")

## 9. Results: Internal vs External R²

In [ ]:
# Display full results
print("="*80)
print("EXTERNAL VALIDATION RESULTS")
print("="*80)
print(df_external_results.to_string(index=False))

# Get best probe for each descriptor
print("\n" + "="*80)
print("BEST PROBE PER DESCRIPTOR")
print("="*80)

best_results = df_external_results.loc[df_external_results.groupby('descriptor')['r2_external'].idxmax()]
print(best_results[['descriptor', 'probe_type', 'r2_internal', 'r2_external', 'r2_drop']].to_string(index=False))

## 10. Visualization: Internal vs External Performance

In [ ]:
# Bar plot: Internal R² vs External R² (best probe for each descriptor)
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(target_descriptors))
width = 0.35

r2_internal = best_results['r2_internal'].values
r2_external = best_results['r2_external'].values

bars1 = ax.bar(x - width/2, r2_internal, width, label='Internal Test (MassSpecGym)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, r2_external, width, label='External Validation (MassBank EU)', color='#e74c3c', alpha=0.8)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Descriptor', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('SSL Embeddings: Internal vs External Validation Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(target_descriptors, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=0.5, color='green', linestyle='--', linewidth=1, alpha=0.5, label='R²=0.5 threshold')

plt.tight_layout()
plt.show()

In [ ]:
# R² drop analysis
fig, ax = plt.subplots(figsize=(14, 6))

r2_drops = best_results['r2_drop'].values
colors = ['#e74c3c' if drop > 0.1 else '#f39c12' if drop > 0.05 else '#2ecc71' for drop in r2_drops]

bars = ax.bar(target_descriptors, r2_drops, color=colors, alpha=0.7, edgecolor='black')

# Add value labels
for bar, drop in zip(bars, r2_drops):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{drop:.3f}', ha='center', va='bottom' if drop > 0 else 'top', fontsize=10, fontweight='bold')

ax.set_xlabel('Descriptor', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Drop (Internal - External)', fontsize=12, fontweight='bold')
ax.set_title('Generalization Gap: R² Drop from Internal to External Test', fontsize=14, fontweight='bold')
ax.set_xticklabels(target_descriptors, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axhline(y=0.05, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='5% threshold')
ax.axhline(y=0.10, color='red', linestyle='--', linewidth=1, alpha=0.5, label='10% threshold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Legend
print("\n🎨 Color coding:")
print("   🟢 Green:  Drop ≤ 0.05 (excellent generalization)")
print("   🟠 Orange: Drop 0.05-0.10 (good generalization)")
print("   🔴 Red:    Drop > 0.10 (moderate domain shift)")

## 11. Summary Statistics

In [ ]:
print("="*80)
print("EXTERNAL VALIDATION SUMMARY")
print("="*80)

mean_r2_internal = best_results['r2_internal'].mean()
mean_r2_external = best_results['r2_external'].mean()
mean_drop = best_results['r2_drop'].mean()
max_drop = best_results['r2_drop'].max()
min_drop = best_results['r2_drop'].min()

print(f"\nInternal Test (MassSpecGym, n={len(df_msg_test):,}):")
print(f"  Mean R²: {mean_r2_internal:.4f}")
print(f"  Range:   {best_results['r2_internal'].min():.4f} - {best_results['r2_internal'].max():.4f}")

print(f"\nExternal Validation (MassBank EU, n={len(df_massbank)}):")
print(f"  Mean R²: {mean_r2_external:.4f}")
print(f"  Range:   {best_results['r2_external'].min():.4f} - {best_results['r2_external'].max():.4f}")

print(f"\nGeneralization Gap:")
print(f"  Mean R² drop:  {mean_drop:.4f} ({mean_drop/mean_r2_internal*100:.1f}%)")
print(f"  Max R² drop:   {max_drop:.4f} ({target_descriptors[best_results['r2_drop'].argmax()]})")
print(f"  Min R² drop:   {min_drop:.4f} ({target_descriptors[best_results['r2_drop'].argmin()]})")

print(f"\nDescriptors with good generalization (drop < 0.05):")
good_gen = best_results[best_results['r2_drop'] < 0.05]
print(f"  {len(good_gen)}/{len(best_results)} descriptors")
if len(good_gen) > 0:
    print(f"  Descriptors: {', '.join(good_gen['descriptor'].values)}")

print(f"\nDescriptors with moderate shift (drop > 0.10):")
moderate_shift = best_results[best_results['r2_drop'] > 0.10]
print(f"  {len(moderate_shift)}/{len(best_results)} descriptors")
if len(moderate_shift) > 0:
    print(f"  Descriptors: {', '.join(moderate_shift['descriptor'].values)}")

print("\n" + "="*80)

## 12. Save Results

In [ ]:
# Save external validation results
results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)

output_path = results_dir / 'external_validation_massbank.pkl'
with open(output_path, 'wb') as f:
    pickle.dump({
        'full_results': df_external_results.to_dict('records'),
        'best_results': best_results.to_dict('records'),
        'summary': {
            'mean_r2_internal': mean_r2_internal,
            'mean_r2_external': mean_r2_external,
            'mean_drop': mean_drop,
            'n_internal': len(df_msg_test),
            'n_external': len(df_massbank)
        }
    }, f)

print(f"✅ Saved external validation results to: {output_path}")

# Also save as CSV for easy viewing
csv_path = results_dir / 'external_validation_massbank.csv'
best_results.to_csv(csv_path, index=False)
print(f"✅ Saved summary table to: {csv_path}")

## 13. Interpretation for Thesis

### What These Results Tell Us

1. **Small R² drop (< 0.05)**: SSL embeddings encode generalizable molecular features
   - These descriptors are robust to different instruments/databases
   - Strong evidence that SSL learned universal spectral→molecular mapping

2. **Moderate R² drop (0.05-0.10)**: Acceptable generalization with some domain shift
   - Expected for complex descriptors or instrument-dependent patterns
   - Still useful for external datasets, but with caveats

3. **Large R² drop (> 0.10)**: Significant domain gap
   - May indicate overfitting to MassSpecGym characteristics
   - Could require domain adaptation or fine-tuning for external data

### For Your Thesis

Report both metrics:
- **Internal validation**: Performance on same-distribution held-out data
- **External validation**: Generalization to different MS/MS database

Example text:
> "We evaluated the learned SSL representations on both an internal test set (MassSpecGym, n=45,185) and an external validation set from MassBank EU (n=758 unique compounds, deduplicated by InChIKey). The mean R² on internal data was 0.XXX, dropping to 0.YYY on external validation, indicating [excellent/good/moderate] generalization across MS/MS databases."